In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from sklearn.metrics import mutual_info_score
from matplotlib.backends.backend_pdf import PdfPages

# =========================
# Leakage Model
# =========================

def hw(x):
    return np.unpackbits(np.array([x], dtype=np.uint8)).sum()

def leakage(state, noise=1.0, blinding=0.0):
    hw_val = hw(state)
    hd_val = abs(hw_val - np.random.randint(0, 8))
    blind = np.random.normal(0, blinding)
    return 0.8 * hw_val + 0.2 * hd_val + blind + np.random.normal(0, noise)

# =========================
# Trace Generator (Unified)
# =========================

def gen_traces(n, L, key, noise=1.0, masking=False, blinding=0.0, shuffle=False):
    pts = np.random.randint(0, 256, n)
    traces = np.zeros((n, L))

    for i in range(n):
        s = pts[i] ^ key

        # ===== MASKING =====
        if masking:
            r = np.random.randint(0, 256)
            s_masked = s ^ r
            leak = leakage(np.uint8(s_masked), noise, 0.0)
        else:
            leak = leakage(np.uint8(s), noise, 0.0)

        # ===== BLINDING =====
        leak = leak + np.random.normal(0, blinding)

        trace = leak + np.random.normal(0, 0.5, L)

        if shuffle:
            np.random.shuffle(trace)

        traces[i] = trace

    return traces, pts

# =========================
# CPA (optimized)
# =========================

def cpa(traces, pts):
    n, L = traces.shape
    X = (traces - traces.mean(0)) / (traces.std(0) + 1e-12)

    best_k, best_score = 0, -1
    corr_matrix = np.zeros((256, L))

    for k in range(256):
        hyp = np.array([hw(p ^ k) for p in pts])
        hyp = (hyp - hyp.mean()) / (hyp.std() + 1e-12)

        corr = (hyp @ X) / (n - 1)
        corr_matrix[k] = corr

        score = np.max(np.abs(corr))
        if score > best_score:
            best_score = score
            best_k = k

    return best_k, best_score, corr_matrix

# =========================
# TVLA
# =========================

def tvla(traces, pts):
    g0 = traces[(pts & 1) == 0]
    g1 = traces[(pts & 1) == 1]

    return (g0.mean(0) - g1.mean(0)) / np.sqrt(
        g0.var(0)/len(g0) + g1.var(0)/len(g1) + 1e-12
    )

# =========================
# MI
# =========================

def mi(traces, pts):
    return np.array([
        mutual_info_score(np.round(traces[:, i]*10).astype(int), pts)
        for i in range(traces.shape[1])
    ])

# =========================
# Heatmap PDF
# =========================

def save_heatmap(mat, name, n, tag):
    plt.figure(figsize=(10,4))
    plt.imshow(np.abs(mat), aspect="auto")
    plt.colorbar()
    plt.title(f"{tag} - {name} - N={n}")
    plt.tight_layout()
    plt.savefig(f"{tag}_heatmap_{name}_{n}.pdf")
    plt.close()

# =========================
# Experiment Runner
# =========================

def run():
    key = 42
    Ns = [200, 1000, 5000, 20000]

    defenses = {
        "MASKING": {"masking": True, "blinding": 0.0, "shuffle": False},
        "BLINDING": {"masking": False, "blinding": 3.0, "shuffle": False},
        "SHUFFLE": {"masking": False, "blinding": 0.0, "shuffle": True},
        "NONE": {"masking": False, "blinding": 0.0, "shuffle": False},
        "COMBINED": {"masking": True, "blinding": 3.0, "shuffle": True},
    }

    results = {}

    for name, cfg in defenses.items():
        results[name] = []

        for n in Ns:
            traces, pts = gen_traces(
                n, 80, key,
                masking=cfg["masking"],
                blinding=cfg["blinding"],
                shuffle=cfg["shuffle"]
            )

            k_hat, cpa_score, corr_matrix = cpa(traces, pts)
            tvals = tvla(traces, pts)
            mi_vals = mi(traces, pts)

            results[name].append({
                "n": n,
                "success": int(k_hat == key),
                "cpa": cpa_score,
                "tvla": np.max(np.abs(tvals)),
                "mi": np.mean(mi_vals)
            })

            # Heatmap per point
            save_heatmap(corr_matrix, name, n, "CPA")

            print(f"{name} | n={n}")

    return results

# =========================
# PDF PLOTS
# =========================

def save_pdf(results, metric, filename):
    with PdfPages(filename) as pdf:
        plt.figure()

        for name, vals in results.items():
            x = [v["n"] for v in vals]
            y = [v[metric] for v in vals]
            plt.plot(x, y, marker="o", label=name)

        plt.xscale("log")
        plt.title(metric.upper() + " comparison")
        plt.legend()

        pdf.savefig()
        plt.close()

# =========================
# RUN
# =========================

results = run()

save_pdf(results, "cpa", "CPA.pdf")
save_pdf(results, "tvla", "TVLA.pdf")
save_pdf(results, "mi", "MI.pdf")

print("DONE")


MASKING | n=200
MASKING | n=1000
MASKING | n=5000
MASKING | n=20000
BLINDING | n=200
BLINDING | n=1000
BLINDING | n=5000
BLINDING | n=20000
SHUFFLE | n=200
SHUFFLE | n=1000
SHUFFLE | n=5000
SHUFFLE | n=20000
NONE | n=200
NONE | n=1000
NONE | n=5000
NONE | n=20000
COMBINED | n=200
COMBINED | n=1000
COMBINED | n=5000
COMBINED | n=20000
DONE


In [11]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mutual_info_score
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd

# =========================
# Leakage Model
# =========================

def hw(x):
    return np.unpackbits(np.array([x], dtype=np.uint8)).sum()

def leakage(state, noise=1.0, blinding=0.0):
    hw_val = hw(state)
    hd_val = abs(hw_val - np.random.randint(0, 8))
    blind = np.random.normal(0, blinding)
    return 0.8 * hw_val + 0.2 * hd_val + blind + np.random.normal(0, noise)

# =========================
# Trace Generator
# =========================

def gen_traces(n, L, key, noise=1.0, masking=False, blinding=0.0, shuffle=False):
    pts = np.random.randint(0, 256, n)
    traces = np.zeros((n, L))

    for i in range(n):
        s = pts[i] ^ key

        if masking:
            r = np.random.randint(0, 256)
            s = s ^ r

        leak = leakage(np.uint8(s), noise, blinding)
        trace = leak + np.random.normal(0, 0.5, L)

        if shuffle:
            np.random.shuffle(trace)

        traces[i] = trace

    return traces, pts

# =========================
# CPA
# =========================

def hw_vec(x):
    return np.array([hw(p ^ x) for p in pts_global])

def cpa(traces, pts):
    global pts_global
    pts_global = pts

    n, L = traces.shape
    X = (traces - traces.mean(0)) / (traces.std(0) + 1e-12)

    best_k, best_score = 0, -1
    corr_matrix = np.zeros((256, L))

    for k in range(256):
        hyp = hw_vec(k)
        hyp = (hyp - hyp.mean()) / (hyp.std() + 1e-12)

        corr = (hyp @ X) / (n - 1)
        corr_matrix[k] = corr

        score = np.max(np.abs(corr))
        if score > best_score:
            best_score = score
            best_k = k

    return best_k, best_score, corr_matrix

# =========================
# TVLA (Welch t-test)
# =========================

def tvla(traces, pts):
    g0 = traces[(pts & 1) == 0]
    g1 = traces[(pts & 1) == 1]

    mean_diff = g0.mean(0) - g1.mean(0)
    var = g0.var(0)/len(g0) + g1.var(0)/len(g1) + 1e-12

    return mean_diff / np.sqrt(var)

# =========================
# Mutual Information
# =========================

def mi(traces, pts):
    return np.array([
        mutual_info_score(np.round(traces[:, i]*10).astype(int), pts)
        for i in range(traces.shape[1])
    ])

# =========================
# Heatmap
# =========================

def save_heatmap(mat, name, n, tag):
    plt.figure(figsize=(10,4))
    plt.imshow(np.abs(mat))
    plt.colorbar()
    plt.title(f"{tag}-{name}-N={n}")
    plt.tight_layout()
    plt.savefig(f"{tag}_heatmap_{name}_{n}.pdf")
    plt.close()

# =========================
# RUN EXPERIMENT
# =========================

def run():
    key = 42
    Ns = [200, 1000, 5000, 20000]

    defenses = {
        "NONE": {"masking": False, "blinding": 0.0, "shuffle": False},
        "MASKING": {"masking": True, "blinding": 0.0, "shuffle": False},
        "BLINDING": {"masking": False, "blinding": 3.0, "shuffle": False},
        "SHUFFLE": {"masking": False, "blinding": 0.0, "shuffle": True},
        "COMBINED": {"masking": True, "blinding": 3.0, "shuffle": True},
    }

    results = []

    for name, cfg in defenses.items():

        for n in Ns:
            traces, pts = gen_traces(
                n, 80, key,
                masking=cfg["masking"],
                blinding=cfg["blinding"],
                shuffle=cfg["shuffle"]
            )

            k_hat, cpa_score, corr_matrix = cpa(traces, pts)
            tvals = tvla(traces, pts)
            mi_vals = mi(traces, pts)

            row = {
                "Defense": name,
                "N": n,
                "KeyFound": int(k_hat == key),
                "CPA": cpa_score,
                "TVLA": np.max(np.abs(tvals)),
                "MI": np.mean(mi_vals)
            }

            results.append(row)

            print(row)

            save_heatmap(corr_matrix, name, n, "CPA")

    return pd.DataFrame(results)

# =========================
# PDF PLOTS
# =========================

def save_pdf(df, metric, filename):
    with PdfPages(filename) as pdf:
        plt.figure()

        for name in df["Defense"].unique():
            sub = df[df["Defense"] == name]
            plt.plot(sub["N"], sub[metric], marker="o", label=name)

        plt.xscale("log")
        plt.title(metric)
        plt.legend()

        pdf.savefig()
        plt.close()

# =========================
# RUN
# =========================

df = run()

# =========================
# SAVE EXCEL
# =========================

df.to_excel("results.xlsx", index=False)

# =========================
# SAVE PDFs
# =========================

save_pdf(df, "CPA", "CPA.pdf")
save_pdf(df, "TVLA", "TVLA.pdf")
save_pdf(df, "MI", "MI.pdf")

print("\nFINAL SUMMARY:\n")
print(df)

{'Defense': 'NONE', 'N': 200, 'KeyFound': 1, 'CPA': 0.747780057687055, 'TVLA': 3.281153661384521, 'MI': 3.5539407970828876}
{'Defense': 'NONE', 'N': 1000, 'KeyFound': 1, 'CPA': 0.7381863920594423, 'TVLA': 8.817817936165598, 'MI': 2.735584471367614}
{'Defense': 'NONE', 'N': 5000, 'KeyFound': 1, 'CPA': 0.7196428005139371, 'TVLA': 20.434932935895333, 'MI': 1.5102979660409581}
{'Defense': 'NONE', 'N': 20000, 'KeyFound': 1, 'CPA': 0.7158049718193963, 'TVLA': 37.92082759707126, 'MI': 0.7656111803248395}
{'Defense': 'MASKING', 'N': 200, 'KeyFound': 0, 'CPA': 0.18326039185071322, 'TVLA': 1.25274997857896, 'MI': 3.546933379769018}
{'Defense': 'MASKING', 'N': 1000, 'KeyFound': 0, 'CPA': 0.11666636287956356, 'TVLA': 1.6668390644195907, 'MI': 2.7232284886108435}
{'Defense': 'MASKING', 'N': 5000, 'KeyFound': 0, 'CPA': 0.039097165494739204, 'TVLA': 1.6594391285408887, 'MI': 1.4109781749287797}
{'Defense': 'MASKING', 'N': 20000, 'KeyFound': 0, 'CPA': 0.019029882558950877, 'TVLA': 1.4123434869542795, 